# Experiment: Kuramoto 0321 Whitened Koopman Chain

本 notebook 用一个 Kuramoto 两团实验展示完整研究链条。目标是：把理论路径、公式、代码结构和结果输出放在同一个文件里，便于后续直接复用。

主链条如下：

1. 生成微观时间序列数据；
2. 固定时间滞后 $\tau$，构造配对样本 $(x_t, x_{t+\tau})$；
3. 选择观测函数并 lift，得到 $X_t = \chi_0(x_t)$ 和 $Y_t = \chi_1(x_{t+\tau})$；
4. 对观测变量中心化，得到 $\tilde X_t, \tilde Y_t$；
5. 计算
$$
C_{00} = \mathbb E[\tilde X_t \tilde X_t^\top], \qquad
C_{01} = \mathbb E[\tilde X_t \tilde Y_t^\top], \qquad
C_{11} = \mathbb E[\tilde Y_t \tilde Y_t^\top];
$$
6. 构造原始回归矩阵
$$
K_{\mathrm{raw}} = C_{00}^{\dagger} C_{01}
$$
以及双边白化矩阵
$$
\bar K = C_{00}^{-1/2} C_{01} C_{11}^{-1/2};
$$
7. 由 $\bar K$ 计算
$$
D_K = (I - \bar K^\top \bar K)^{-1}, \qquad
N_K = \bar K (I - \bar K^\top \bar K)^{-1} \bar K^\top;
$$
8. 计算补充主总分
$$
\mathcal G_\alpha^K = \left(\frac12 - \frac{\alpha}{4}\right) \log \operatorname{pdet}(N_K)
+ \frac{\alpha}{4} \log \operatorname{pdet}(D_K);
$$
9. 对 $K_{\mathrm{raw}}$ 和 $\bar K$ 做奇异值分解，其中 $K_{\mathrm{raw}}$ 只用于奇异值谱对比；
10. 只对 $\bar K$ 的奇异值谱计算 EC 序列和 EC 值；
11. 按定义 12 计算近似可逆性总量
$$
\Gamma_\alpha^K(\bar K)=\sum_{i=1}^{d_{\mathrm{eff}}}\sigma_i^\alpha, \qquad
\gamma_\alpha^K(\bar K)=\frac{1}{d_{\mathrm{eff}}}\Gamma_\alpha^K(\bar K);
$$
12. 按定义 13 计算不同截断维数下的宏观效率增益
$$
\Delta \Gamma_\alpha^K(r)
= \frac{1}{r}\sum_{i=1}^{r}\sigma_i^\alpha
- \frac{1}{d_{\mathrm{eff}}}\sum_{i=1}^{d_{\mathrm{eff}}}\sigma_i^\alpha;
$$
13. 用 $\bar K$ 的前 $r$ 个左奇异向量构造粗粒化函数和宏观变量。

这个 notebook 强调两点：

- $\bar K$ 是主研究对象，$K_{\mathrm{raw}}$ 只用于对照；
- 数据生成部分尽量独立，后面的分析块尽量复用。


## Block 1. 环境导入与统一风格

下面的代码块负责：

- 导入需要的库；
- 把仓库根目录加入 `sys.path`；
- 导入已有工具函数；
- 设置统一绘图风格；
- 定义少量 notebook 本地可视化辅助函数。

统一风格约定：

- 热力图默认使用 `vlag`；
- 同类图尽量保持同一种配色；
- 微观代表曲线只画少量变量，避免过密；
- 只在 notebook 中定义轻量辅助函数，不把实验专用绘图逻辑塞进 `tools.py`。


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = None
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / 'tools' / 'tools.py').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repository root containing tools/tools.py')
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
import pysindy as ps

from tools import (
    compute_transition_covariances,
    fit_data_koopman_operator,
    whiten_operator_matrix,
    get_positive_contributions,
    compute_entropy,
    init_artifacts,
    analyze_kbar_metrics,
    build_macro_from_kbar,
    print_summary,
    compute_gamma_ce_metrics,
)
from data.data_func import plot_clustered_kuramoto

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 160
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = [
    'Microsoft YaHei',
    'SimHei',
    'Noto Sans CJK SC',
    'Source Han Sans SC',
    'Arial Unicode MS',
    'DejaVu Sans',
]

HEATMAP_CMAP = 'vlag'


def sparse_labels(labels, step=1):
    if labels is None:
        return False
    if step <= 1:
        return labels
    return [label if idx % step == 0 else '' for idx, label in enumerate(labels)]


def plot_matrix_heatmap(matrix, title, row_labels=None, col_labels=None, center=0.0, figsize=(6, 6), label_step=1, cmap=HEATMAP_CMAP):
    matrix = np.asarray(matrix)
    n_rows, n_cols = matrix.shape
    is_square = n_rows == n_cols

    if is_square:
        plot_size = (6, 6)
        square = True
        x_step = max(label_step, max(1, int(np.ceil(n_cols / 10))))
        y_step = max(label_step, max(1, int(np.ceil(n_rows / 10))))
    else:
        plot_size = figsize
        square = False
        x_step = max(label_step, max(1, int(np.ceil(n_cols / 8))))
        y_step = max(label_step, max(1, int(np.ceil(n_rows / 18))))

    fig, ax = plt.subplots(figsize=plot_size)
    sns.heatmap(
        matrix,
        ax=ax,
        cmap=cmap,
        center=center,
        xticklabels=sparse_labels(col_labels, x_step),
        yticklabels=sparse_labels(row_labels, y_step),
        square=square,
        cbar_kws={'shrink': 0.72, 'fraction': 0.035, 'pad': 0.02},
    )
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=90, labelsize=6.5)
    ax.tick_params(axis='y', rotation=0, labelsize=6.5)
    plt.tight_layout()
    plt.show()


def standardize_for_plot(x):
    x = np.asarray(x, dtype=float)
    return (x - np.mean(x)) / (np.std(x) + 1e-12)


## Block 2. 全局参数配置

这里集中设置实验参数，不把关键超参数写死在后续单元里。

本实验的关键参数包括：

- Kuramoto 两团系统的数据参数；
- 数据整理参数，如 `burn_in_steps` 和 `sample_stride`；
- 主链条参数，如 `tau`、`lag_steps`、`alpha`、`eps`、`ridge`；
- 观测函数选择；
- 手动指定的截断维数 `rank`；
- 可视化时展示的代表性微观变量索引。

注意：这里的 `tau` 是“物理时间滞后”，而实际配对时的步长由 `lag_steps` 决定，因此一般有
$$
\tau = \text{lag\_steps} \times \Delta t_{\text{fit}}.
$$


In [ ]:
config = {
    'experiment_name': 'exp_kuramoto_0321',
    'kuramoto': {
        'N': 10,
        'n_clusters': 2,
        'K_intra': 5.0,
        'K_inter': 0.07,
        'noise': 0.0,
        'T': 100.0,
        'dt': 0.01,
        'random_seed1': 0,
        'random_seed2': 0,
    },
    'burn_in_steps': 1000,
    'sample_stride': 10,  #15
    'lag_steps': 12, # tuned
    'rank': 2,
    'macro_r_mode': 'selected',
    'alpha': 1.0,
    'eps': 1e-10,
    'ridge': 1e-10,
    'observable_mode': 'identity + fourier', #手动
    'rank_candidates': None,
    'selected_micro_indices': [0, 5],
    'heatmap_label_step': 1,
    'top_k_features_to_show': 5,
}

artifacts = init_artifacts(config)
config


## Block 3. 数据生成：Kuramoto 两团系统

下面的代码块调用现有的 Kuramoto 两团生成逻辑，参考已有 notebook 中 `plot_clustered_kuramoto` 的部分。

这里输出的是微观时间序列数据。对当前实验而言，原始状态使用的是 `sin/cos` 嵌入后的序列，因此原始数据已经是一个二维数组：
$$
X_{\mathrm{raw}} \in \mathbb R^{T \times d}.
$$

我们保留：

- 微观轨道 `x_data_raw`；
- 相位历史 `theta_hist`；
- 时间轴 `t_data_raw`；
- 耦合矩阵 `K_matrix_true`。

这一块是后续最方便替换的部分。以后做实验 2 时，理想情况下只换这里。


In [ ]:
x_data_raw, theta_hist, t_data_raw, K_matrix_true = plot_clustered_kuramoto(**config['kuramoto'])

N = config['kuramoto']['N']
state_names_raw = [f'cos_theta_{i}' for i in range(N)] + [f'sin_theta_{i}' for i in range(N)]

artifacts['raw'] = {
    'x_data_raw': x_data_raw,
    'theta_hist': theta_hist,
    't_data_raw': t_data_raw,
    'K_matrix_true': K_matrix_true,
    'state_names_raw': state_names_raw,
}

print('Raw data shape:', x_data_raw.shape)
print('Theta history shape:', theta_hist.shape)
print('Time axis length:', len(t_data_raw))
print('State dimension:', x_data_raw.shape[1])
print('K matrix shape:', K_matrix_true.shape)

plt.figure(figsize=(12, 4))
for idx in config['selected_micro_indices']:
    plt.plot(t_data_raw, x_data_raw[:, idx], linewidth=1.0, label=state_names_raw[idx])
plt.title('Micro trajectories before preparation')
plt.xlabel('Time')
plt.ylabel('State value')
plt.legend()
plt.tight_layout()
plt.show()


## Block 4. 数据整理：不是强制标准化，而是统一输入格式

这个块的主要目的是 **data preparation**，不是强制标准化。

它负责：

- 去掉 burn-in；
- 进行可选下采样；
- 把数据整理成后续协方差估计和 Koopman 分析所需的统一格式；
- 输出 `x_data_fit`、`t_data_fit` 和 `state_names_fit`。

为什么需要这一步？

1. 不同实验的数据生成函数输出格式可能不同；
2. 主链条后面希望只依赖统一格式的数据；
3. 我们通常不希望把过渡阶段直接拿来估计 $C_{00}, C_{01}, C_{11}$；
4. 下采样可以减少冗余样本，让后面的可视化和矩阵计算更清晰。

因此，这一步更准确地说是“数据整理与统一”，而不是必须做 z-score 或 min-max 标准化。


In [ ]:
burn_in_steps = config['burn_in_steps']
sample_stride = config['sample_stride']

x_data_fit = x_data_raw[burn_in_steps::sample_stride].copy()
t_data_fit = t_data_raw[burn_in_steps::sample_stride].copy()
state_names_fit = state_names_raw.copy()
dt_fit = config['kuramoto']['dt'] * sample_stride
tau = config['lag_steps'] * dt_fit

artifacts['prep'] = {
    'x_data_fit': x_data_fit,
    't_data_fit': t_data_fit,
    'state_names_fit': state_names_fit,
    'dt_fit': dt_fit,
    'tau': tau,
}

print('Burn-in steps removed:', burn_in_steps)
print('Sample stride:', sample_stride)
print('Prepared data shape:', x_data_fit.shape)
print('Prepared dt:', dt_fit)
print('tau = lag_steps * dt_fit =', tau)

plt.figure(figsize=(12, 4))
for idx in config['selected_micro_indices']:
    plt.plot(t_data_fit, x_data_fit[:, idx], linewidth=1.0, label=state_names_fit[idx])
plt.title('Micro trajectories after preparation')
plt.xlabel('Time')
plt.ylabel('State value')
plt.legend()
plt.tight_layout()
plt.show()


## Block 5. 观测函数与 lift

现在选择观测函数 $\chi$，把原始微观数据映射到观测空间。

在本 notebook 中，这块先不封装成统一函数，直接在单元里写，保持实验特异性。候选的观测函数包括：

- `IdentityLibrary()`；
- `FourierLibrary()`；
- `PolynomialLibrary()`；
- `CustomLibrary()`。

本次实验的默认选择是
$$
\chi(x) = x,
$$
也就是 identity observable。后续换实验时，只要替换这一块，后面的主链条尽量可以继续复用。


In [ ]:
ide = ps.IdentityLibrary()
fourier = ps.FourierLibrary(n_frequencies=1)
ode_lib = ps.PolynomialLibrary(degree=2, include_bias=False)
library_functions = [
    lambda x: 1,
    lambda x: x,
    lambda x: np.sin(x),
    lambda x: np.cos(x),
]
custom_library = ps.CustomLibrary(library_functions=library_functions)

library = ide + fourier
library.fit(x_data_fit)
x_data_lift = np.asarray(library.transform(x_data_fit), dtype=float)
raw_names = library.get_feature_names(input_features=state_names_fit)

x_data_lift_centered = np.asarray(x_data_lift - np.mean(x_data_lift, axis=0, keepdims=True), dtype=float)

artifacts['obs'] = {
    'library': library,
    'x_data_lift': x_data_lift,
    'x_data_lift_centered': x_data_lift_centered,
    'feature_names': raw_names,
    'observable_mode': config['observable_mode'],
}

# x_data_lift = x_data_lift_centered # 不需要减去均值就注释掉，需要减去均值就正常保留

print('Observable mode:', config['observable_mode'])
print('Input shape:', x_data_fit.shape)
print('Lifted shape:', x_data_lift.shape)
print('First 10 feature names:', raw_names[:10])


## Block 6. 计算 $C_{00}$、$C_{01}$、$C_{11}$

这一块是研究链条的核心统计层。

我们根据中心化后的观测变量构造：

$$
C_{00} = \mathbb E[\tilde X_t \tilde X_t^\top],
\qquad
C_{01} = \mathbb E[\tilde X_t \tilde Y_t^\top],
\qquad
C_{11} = \mathbb E[\tilde Y_t \tilde Y_t^\top].
$$

其中当前与未来样本通过 `lag_steps` 对齐。这里统一调用已有工具函数 `compute_transition_covariances(...)`。


In [ ]:
transition_stats = compute_transition_covariances(
    [x_data_lift],
    library=None,
    weights='uniform',
    lag_steps=config['lag_steps'],
)

C00 = transition_stats['C00']
C01 = transition_stats['C01']
C11 = transition_stats['C11']
X_pairs = transition_stats['X']
Y_pairs = transition_stats['Y']

artifacts['cov'] = {
    'transition_stats': transition_stats,
    'C00': C00,
    'C01': C01,
    'C11': C11,
    'X_pairs': X_pairs,
    'Y_pairs': Y_pairs,
}

print(f"lag_steps = {config['lag_steps']}, tau = {tau:.6f}")
print('Pair count:', X_pairs.shape[0])
print('C00 shape:', C00.shape)
print('C01 shape:', C01.shape)
print('C11 shape:', C11.shape)
print('cond(C00):', np.linalg.cond(C00))
print('cond(C11):', np.linalg.cond(C11))
print('min eig(C00):', np.min(np.linalg.eigvalsh(0.5 * (C00 + C00.T))))
print('min eig(C11):', np.min(np.linalg.eigvalsh(0.5 * (C11 + C11.T))))

plot_matrix_heatmap(C00, 'C00', row_labels=raw_names, col_labels=raw_names, label_step=config['heatmap_label_step'])
plot_matrix_heatmap(C01, 'C01', row_labels=raw_names, col_labels=raw_names, label_step=config['heatmap_label_step'])
plot_matrix_heatmap(C11, 'C11', row_labels=raw_names, col_labels=raw_names, label_step=config['heatmap_label_step'])


## Block 7. 构造 $K_{\mathrm{raw}}$ 与 $\bar K$

根据协方差矩阵，原始回归矩阵定义为
$$
K_{\mathrm{raw}} = C_{00}^{\dagger} C_{01},
$$
而双边白化矩阵定义为
$$
\bar K = C_{00}^{-1/2} C_{01} C_{11}^{-1/2}.
$$

在这里：

- `K_raw` 只用于后续奇异值谱对比；
- `K_bar` 才是主分析对象。

统一调用 `fit_data_koopman_operator(...)`，它会返回：

- 数据驱动的一步算子 `A`；
- 白化矩阵 `K_bar`；
- 白化所需的平方根与逆平方根矩阵。


In [ ]:
koop_fit = fit_data_koopman_operator(
    [x_data_lift],
    library=None,
    weights='uniform',
    eps=config['eps'],
    ridge=config['ridge'],
    lag_steps=config['lag_steps'],
)

K_raw = koop_fit['A']
K_bar = koop_fit['K_bar']
K_bar_from_A = koop_fit['K_bar_from_A']
C00_inv_sqrt = koop_fit['C00_inv_sqrt']
C11_inv_sqrt = koop_fit['C11_inv_sqrt']

artifacts['koopman'] = {
    'koop_fit': koop_fit,
    'K_raw': K_raw,
    'K_bar': K_bar,
    'K_bar_from_A': K_bar_from_A,
    'C00_inv_sqrt': C00_inv_sqrt,
    'C11_inv_sqrt': C11_inv_sqrt,
}

print('K_raw shape:', K_raw.shape)
print('K_bar shape:', K_bar.shape)
print('||K_bar - K_bar_from_A||_F =', np.linalg.norm(K_bar - K_bar_from_A))
print('sigma_max(K_bar) <= 1, approx =', np.linalg.svd(K_bar, compute_uv=False)[0])

plot_matrix_heatmap(K_raw, 'K_raw', row_labels=raw_names, col_labels=raw_names, label_step=config['heatmap_label_step'])
plot_matrix_heatmap(K_bar, 'K_bar', row_labels=raw_names, col_labels=raw_names, label_step=config['heatmap_label_step'])



## Block 8. 奇异值谱分析与 EC 指标

对白化矩阵做奇异值分解：

$$
\bar K = U \Sigma V^\top,
$$

同时对 `K_raw` 做奇异值分解，但 `K_raw` 只作为对照。这里共绘制四张谱图：

1. `K_bar` 的奇异值谱；
2. `K_raw` 的奇异值谱；
3. `K_bar` 与 `K_raw` 的整体对比图；
4. 二者前 10% 奇异值的双坐标对比图。

随后仅基于 `K_bar` 的奇异值谱计算 EC 序列与 EC 值。


In [ ]:

U_raw, S_raw, Vt_raw = np.linalg.svd(K_raw, full_matrices=False)
metrics = analyze_kbar_metrics(K_bar, alpha=config['alpha'], eps=config['eps'])

U_bar = metrics['U']
S_bar = metrics['S']
Vt_bar = metrics['Vt']

diff = get_positive_contributions(S_bar)
EC = compute_entropy(diff)

top10_count = max(1, int(np.ceil(0.1 * len(S_bar))))

artifacts['spectral'] = {
    'U_raw': U_raw,
    'S_raw': S_raw,
    'Vt_raw': Vt_raw,
    'U_bar': U_bar,
    'S_bar': S_bar,
    'Vt_bar': Vt_bar,
    'diff': diff,
    'EC': EC,
    'top10_count': top10_count,
}
artifacts['metrics']['kbar'] = metrics
artifacts['metrics']['EC'] = EC
artifacts['metrics']['EC_diff'] = diff

print('Top 10 singular values of K_bar:', S_bar[:10])
print('Top 10 singular values of K_raw:', S_raw[:10])
print('EC entropy:', EC)
print('Top 10% singular-value count:', top10_count)

idx_bar = np.arange(1, len(S_bar) + 1)
idx_raw = np.arange(1, len(S_raw) + 1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(idx_bar, S_bar, marker='o', linewidth=1.8, label='K_bar singular values', color='tab:blue')
ax.axhline(1.0, color='gray', linestyle=':', linewidth=1.0)
ax.set_xlabel('Singular index')
ax.set_ylabel('Singular value')
ax.set_title('Singular spectrum: K_bar')
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(idx_raw, S_raw, marker='s', linewidth=1.4, linestyle='--', label='K_raw singular values', color='0.35')
ax.set_xlabel('Singular index')
ax.set_ylabel('Singular value')
ax.set_title('Singular spectrum: K_raw')
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(idx_bar, S_bar, marker='o', linewidth=1.8, label='K_bar singular values', color='tab:blue')
ax.plot(idx_raw, S_raw, marker='s', linewidth=1.2, linestyle='--', label='K_raw singular values', color='0.35')
ax.axhline(1.0, color='gray', linestyle=':', linewidth=1.0)
ax.set_xlabel('Singular index')
ax.set_ylabel('Singular value')
ax.set_title('Singular spectrum comparison: K_bar vs K_raw')
ax.legend()
plt.tight_layout()
plt.show()

fig, ax_left = plt.subplots(figsize=(8, 4.5))
ax_right = ax_left.twinx()
line1 = ax_left.plot(idx_bar[:top10_count], S_bar[:top10_count], marker='o', linewidth=1.8, label=f'K_bar top 10% ({top10_count} values)', color='tab:purple')
line2 = ax_right.plot(idx_raw[:top10_count], S_raw[:top10_count], marker='s', linewidth=1.2, linestyle='--', label=f'K_raw top 10% ({top10_count} values)', color='0.35')
ax_left.axhline(1.0, color='tab:purple', linestyle=':', linewidth=1.0, alpha=0.6)
ax_left.set_xlabel('Singular index')
ax_left.set_ylabel('K_bar singular value', color='tab:purple')
ax_right.set_ylabel('K_raw singular value', color='0.35')
ax_left.tick_params(axis='y', colors='tab:purple')
ax_right.tick_params(axis='y', colors='0.35')
ax_left.set_ylim(0.0, 1.0)
ax_left.set_title('Top 10% singular spectrum comparison')
lines = line1 + line2
labels = [line.get_label() for line in lines]
ax_left.legend(lines, labels, loc='best')
plt.tight_layout()
plt.show()


## Block 9. 计算 $D_K$、$N_K$ 与主总分 $\mathcal G_\alpha^K$

根据主链条，

$$
D_K = (I - \bar K^\top \bar K)^{-1},
\qquad
N_K = \bar K (I - \bar K^\top \bar K)^{-1} \bar K^\top.
$$

并定义主总分

$$
\mathcal G_\alpha^K = \left(\frac12 - \frac{\alpha}{4}\right) \log \operatorname{pdet}(N_K)
+ \frac{\alpha}{4} \log \operatorname{pdet}(D_K).
$$

这一块只围绕 $\bar K$ 展开，因为它才是主研究对象。边缘修正项在这个 notebook 中不作为主链条核心，只保留为以后可扩展的附加项。


In [ ]:
D_K = metrics['D_K']
N_K = metrics['N_K']
G_alpha_K = metrics['G_alpha_K']

print('log pdet(D_K):', metrics['log_pdet_D_K'])
print('log pdet(N_K):', metrics['log_pdet_N_K'])
print('G_alpha^K:', G_alpha_K)

plot_matrix_heatmap(D_K, 'D_K', center=None, figsize=(6, 6), cmap=HEATMAP_CMAP)
plot_matrix_heatmap(N_K, 'N_K', center=0.0, figsize=(6, 6), cmap=HEATMAP_CMAP)

plt.figure(figsize=(8, 3.8))
plt.bar(np.arange(1, len(diff) + 1), diff, color='tab:blue', alpha=0.85)
plt.title('EC increments derived from K_bar singular values')
plt.xlabel('Index')
plt.ylabel('Increment')
plt.tight_layout()
plt.show()



## Block 10. 定义 12/13 下的可逆性、维度平均可逆性与 CE

对白化 Koopman 矩阵的奇异值 $\sigma_i$，按定义 12 计算：

$$
\Gamma_\alpha^K(\bar K)=\sum_{i=1}^{d_{\mathrm{eff}}}\sigma_i^\alpha,
\qquad
\gamma_\alpha^K(\bar K)=\frac{1}{d_{\mathrm{eff}}}\Gamma_\alpha^K(\bar K).
$$

再按定义 13，对每个候选截断维数 $r$ 计算：

$$
\Delta \Gamma_\alpha^K(r)
= \frac{1}{r} \sum_{i=1}^{r} \sigma_i^\alpha
- \frac{1}{d_{\mathrm{eff}}} \sum_{i=1}^{d_{\mathrm{eff}}} \sigma_i^\alpha.
$$

这里会同时给出：

- 手动指定 `rank` 对应的 CE；
- 自动选择 `selected_r` 对应的 CE；
- 完整的 $\Delta \Gamma_\alpha^K(r)$ 曲线。


In [ ]:

manual_r = config.get('rank')
rank_candidates = config.get('rank_candidates')
gamma_metrics = compute_gamma_ce_metrics(
    S_bar,
    alpha=config['alpha'],
    rank_candidates=rank_candidates,
    manual_r=manual_r,
    eps=config['eps'],
)

reversibility_channel_scores = gamma_metrics['reversibility_channel_scores']
Gamma_alpha_K = gamma_metrics['Gamma_alpha_K']
gamma_alpha_K = gamma_metrics['gamma_alpha_K']
effective_rank_gamma = gamma_metrics['effective_rank']
rank_candidates = gamma_metrics['rank_candidates']
delta_gamma_by_r = gamma_metrics['delta_gamma_by_r']
selected_r = gamma_metrics['selected_r']
delta_gamma_selected_r = gamma_metrics['delta_gamma_selected_r']
manual_r = gamma_metrics['manual_r']
delta_gamma_manual_r = gamma_metrics['delta_gamma_manual_r']

artifacts['metrics']['reversibility_channel_scores'] = reversibility_channel_scores
artifacts['metrics']['Gamma_alpha_K'] = Gamma_alpha_K
artifacts['metrics']['gamma_alpha_K'] = gamma_alpha_K
artifacts['metrics']['delta_gamma_by_r'] = delta_gamma_by_r
artifacts['metrics']['selected_r'] = selected_r
artifacts['metrics']['delta_gamma_selected_r'] = delta_gamma_selected_r
artifacts['metrics']['delta_g_selected_r'] = delta_gamma_selected_r
artifacts['metrics']['manual_r'] = manual_r
artifacts['metrics']['delta_gamma_manual_r'] = delta_gamma_manual_r

print('Reversibility channel scores sigma_i^alpha:', reversibility_channel_scores)
print('Gamma_alpha_K:', Gamma_alpha_K)
print('gamma_alpha_K:', gamma_alpha_K)
print('effective_rank:', effective_rank_gamma)
print('Delta Gamma by r:', delta_gamma_by_r)
print('Manual r:', manual_r)
print('Delta Gamma at manual r:', delta_gamma_manual_r)
print('Selected r:', selected_r)
print('Delta Gamma at selected r (CE):', delta_gamma_selected_r)

plt.figure(figsize=(7, 4))
plt.plot(list(delta_gamma_by_r.keys()), list(delta_gamma_by_r.values()), marker='o', color='tab:blue')
if manual_r is not None:
    plt.axvline(manual_r, color='tab:orange', linestyle=':', label=f'manual r = {manual_r}')
plt.axvline(selected_r, color='tab:red', linestyle='--', label=f'selected r = {selected_r}')
plt.title('Dimension-averaged reversibility gain Delta Gamma_alpha^K(r)')
plt.xlabel('r')
plt.ylabel('Delta Gamma_alpha^K(r)')
plt.legend()
plt.tight_layout()
plt.show()



## Block 11. 左奇异向量结构可视化

根据研究链条，当前侧粗粒化应选取 $\bar K$ 的左奇异子空间。因此，我们重点可视化：

- 全部左奇异向量矩阵 $U$；
- 左奇异向量绝对值矩阵 $|U|$；
- 用于宏观变量构造的前 $r$ 列左奇异向量；
- 用于宏观变量构造的前 $r$ 列左奇异向量绝对值。

这里的 `r` 可以来自手动指定，也可以来自自动选择；实际采用哪一种，会在代码单元中明确打印。


In [ ]:

macro_r_mode = config.get('macro_r_mode', 'selected')
if macro_r_mode == 'manual' and manual_r is not None:
    macro_r = manual_r
    macro_r_source = f'manual_r = {manual_r}'
else:
    macro_r = selected_r
    macro_r_source = f'selected_r = {selected_r}'

print('Macro variables are constructed using', macro_r_source)

plot_matrix_heatmap(U_bar, 'Left singular vectors U (all)', row_labels=raw_names, col_labels=[f'u{i+1}' for i in range(U_bar.shape[1])], figsize=(7, 8), label_step=config['heatmap_label_step'])
plot_matrix_heatmap(np.abs(U_bar), 'Absolute left singular vectors |U| (all)', row_labels=raw_names, col_labels=[f'|u{i+1}|' for i in range(U_bar.shape[1])], center=None, figsize=(7, 8), label_step=config['heatmap_label_step'], cmap='Blues')
plot_matrix_heatmap(U_bar[:, :macro_r], f'Left singular vectors U (first {macro_r})', row_labels=raw_names, col_labels=[f'u{i+1}' for i in range(macro_r)], figsize=(7, 8), label_step=config['heatmap_label_step'])
plot_matrix_heatmap(np.abs(U_bar[:, :macro_r]), f'Absolute left singular vectors |U| (first {macro_r})', row_labels=raw_names, col_labels=[f'|u{i+1}|' for i in range(macro_r)], center=None, figsize=(7, 8), label_step=config['heatmap_label_step'], cmap='Blues')



## Block 12. 构造粗粒化函数与宏观变量

根据用于宏观表示的前 $r$ 个左奇异向量，构造当前侧宏观变量：

$$
z_t = U_r^\top C_{00}^{-1/2} \tilde X_t.
$$

在行向量约定下，这等价于把每个样本先做白化，再投影到前 $r$ 个左奇异方向上。

这里统一调用 `build_macro_from_kbar(...)`，而 `r` 默认使用自动选出的 `selected_r`，也可以切换为手动指定的 `rank`。


In [ ]:

macro = build_macro_from_kbar(
    U=U_bar,
    S=S_bar,
    Vt=Vt_bar,
    C00_inv_sqrt=C00_inv_sqrt,
    X=X_pairs,
    r=macro_r,
    feature_names=raw_names,
    Y=Y_pairs,
    C11_inv_sqrt=C11_inv_sqrt,
    center=False,
)

artifacts['macro'] = macro
artifacts['metrics']['macro_r'] = macro_r
artifacts['metrics']['macro_r_source'] = macro_r_source

print('Macro current shape:', macro['z_current'].shape)
print('Macro future shape:', None if macro['z_future'] is None else macro['z_future'].shape)
print('Top features per macro component:')
for idx, items in enumerate(macro['dominant_features'], start=1):
    print(f'  Component {idx}:')
    display(pd.DataFrame(items))



## Block 13. 宏观数据与宏微观对比

这一块用来展示：

- 宏观时间序列；
- 微观代表变量的时间序列；
- 宏微观对比图。

这里展示的宏观变量维数由上一块指定的 `macro_r` 决定，并会在图前打印所采用的来源。


In [ ]:

z_current = macro['z_current']
macro_names = [f'z_{i+1}' for i in range(macro_r)]

plt.figure(figsize=(12, 4))
for i in range(macro_r):
    plt.plot(z_current[:, i], linewidth=1.3, label=macro_names[i])
plt.title('Macro time series')
plt.xlabel('Pair index')
plt.ylabel('Macro value')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
for idx in config['selected_micro_indices']:
    plt.plot(standardize_for_plot(X_pairs[:, idx]), linewidth=1.0, alpha=0.8, label=f'micro: {raw_names[idx]}')
for i in range(macro_r):
    plt.plot(standardize_for_plot(z_current[:, i]), linewidth=1.8, linestyle='--', label=f'macro: {macro_names[i]}')
plt.title('Micro vs macro (standardized for visualization only)')
plt.xlabel('Pair index')
plt.ylabel('Standardized value')
plt.legend(ncol=2)
plt.tight_layout()
plt.show()



## Block 14. 统一摘要输出

最后把关键结果统一整理输出，便于快速检查当前实验结果是否完整。

这里重点输出两类信息：

- 结构与矩阵信息：原始维数、提升函数、提升后维数、`D_K/N_K` 形状等；
- 指标信息：`G_alpha_K`、`EC`、`Gamma_alpha_K`、`gamma_alpha_K`、手动 `r`、自动 `selected_r` 及对应 CE。


In [ ]:

summary_dict = {
    'experiment': {
        'name': config['experiment_name'],
        'observable_mode': config['observable_mode'],
        'alpha': config['alpha'],
        'eps': config['eps'],
        'ridge': config['ridge'],
    },
    'data': {
        'raw_shape': x_data_raw.shape,
        'prepared_shape': x_data_fit.shape,
        'lifted_shape': x_data_lift.shape,
        'lift_function': config['observable_mode'],
        'pair_count': X_pairs.shape[0],
        'dt_fit': dt_fit,
        'tau': tau,
        'lag_steps': config['lag_steps'],
    },
    'matrices': {
        'C00_shape': C00.shape,
        'C01_shape': C01.shape,
        'C11_shape': C11.shape,
        'K_raw_shape': K_raw.shape,
        'K_bar_shape': K_bar.shape,
        'D_K_shape': D_K.shape,
        'N_K_shape': N_K.shape,
        'sigma_max_K_bar': metrics['sigma_max'],
        'effective_rank': effective_rank_gamma,
    },
}
print_summary(summary_dict)


In [ ]:

summary_dict = {
    'scores': {
        'G_alpha_K': G_alpha_K,
        'EC': EC,
        'Gamma_alpha_K': Gamma_alpha_K,
        'gamma_alpha_K': gamma_alpha_K,
        'manual_r': manual_r,
        'delta_gamma_manual_r': delta_gamma_manual_r,
        'selected_r': selected_r,
        'delta_gamma_selected_r': delta_gamma_selected_r,
        'macro_r': macro_r,
        'macro_r_source': macro_r_source,
        'top_r_singular_values': S_bar[:macro_r],
    },
}
print_summary(summary_dict)


In [ ]:

summary_dict = {
    'curves': {
        'rank_candidates': rank_candidates,
        'delta_gamma_by_r': delta_gamma_by_r,
    },
}
print_summary(summary_dict)


## Block 15. 本实验专用辅助函数

从这一块开始，内容只服务于当前 Kuramoto 实验，因此统一放在摘要输出之后，而不并入前面的通用主链。

本块新增三类辅助函数：

1. 频谱分析函数：用于计算团序参量与宏观变量的 FFT 频谱，并标记主峰频率；
2. 特征值分解宏观变量函数：对 $\bar K$ 直接做特征值分解，按特征值模长排序，并把特征向量取模长后构造实值宏观变量 $Z_t$；
3. 高斯多元互信息函数：基于协方差矩阵的对数行列式形式，计算连续多维变量之间的互信息。

这里强调两点：

- 频谱部分的微观对象不是单个振子的 `cos/sin` 特征，而是每个团的序参量 $r_c(t)$；
- 特征值分解部分是一个专门的对照实验定义，用来和 SVD 宏观变量比较，它不是前面主链方法的替代。

In [ ]:
def compute_cluster_order_parameter_series(theta, n_clusters):
    theta = np.asarray(theta, dtype=float)
    N = theta.shape[1]
    cluster_size = N // n_clusters
    series = []
    labels = []
    for c in range(n_clusters):
        start = c * cluster_size
        end = N if c == n_clusters - 1 else (c + 1) * cluster_size
        r_c = np.abs(np.mean(np.exp(1j * theta[:, start:end]), axis=1))
        series.append(r_c)
        labels.append(f'Cluster {c + 1}')
    return np.column_stack(series), labels


def compute_fft_spectrum(signal, dt, remove_dc=True, standardize=False):
    signal = np.asarray(signal, dtype=float)
    work = signal.copy()
    if standardize:
        work = (work - np.mean(work)) / (np.std(work) + 1e-12)
    if remove_dc:
        work = work - np.mean(work)
    N_signal = work.shape[0]
    fft_vals = np.fft.fft(work)
    freqs = np.fft.fftfreq(N_signal, d=dt)
    pos_mask = freqs >= 0
    freqs = freqs[pos_mask]
    amps = np.abs(fft_vals[pos_mask]) / N_signal * 2.0
    if freqs.size > 0:
        amps[0] *= 0.5
    return freqs, amps


def mark_peak(ax, freqs, amps, color, text_offset=0.06):
    if len(freqs) <= 1:
        return None
    search_slice = slice(1, len(freqs))
    local_idx = int(np.argmax(amps[search_slice]))
    idx = local_idx + 1
    f_peak = float(freqs[idx])
    a_peak = float(amps[idx])
    ax.scatter(f_peak, a_peak, color=color, zorder=5)
    ax.annotate(
        f'{f_peak:.3f}',
        xy=(f_peak, a_peak),
        xytext=(f_peak, a_peak + text_offset),
        textcoords='data',
        ha='center',
        fontsize=11,
        fontweight='bold',
        color=color,
        arrowprops=dict(arrowstyle='->', color=color, lw=1),
    )
    return {'peak_frequency': f_peak, 'peak_amplitude': a_peak}


def build_macro_from_kbar_eig(K_bar, C00_inv_sqrt, X, r, feature_names=None, center=False, eps=1e-12):
    K_bar = np.asarray(K_bar)
    X = np.asarray(X, dtype=float)
    C00_inv_sqrt = np.asarray(C00_inv_sqrt, dtype=float)

    eigvals, eigvecs = np.linalg.eig(K_bar)
    order = np.argsort(-np.abs(eigvals))
    eigvals_sorted = eigvals[order]
    eigvecs_sorted = eigvecs[:, order]

    W_r = np.abs(eigvecs_sorted[:, :r])
    W_r = W_r / (np.linalg.norm(W_r, axis=0, keepdims=True) + eps)

    X_work = X - np.mean(X, axis=0, keepdims=True) if center else X
    xi = X_work @ C00_inv_sqrt
    z_eig_current = xi @ W_r
    coarse_grain_matrix = C00_inv_sqrt @ W_r

    dominant_features = []
    if feature_names is not None:
        feature_names = list(feature_names)
        for component_idx in range(r):
            weights = np.abs(coarse_grain_matrix[:, component_idx])
            order_idx = np.argsort(-weights)
            top_items = [
                {
                    'feature': feature_names[idx],
                    'weight': float(coarse_grain_matrix[idx, component_idx]),
                    'abs_weight': float(weights[idx]),
                }
                for idx in order_idx[: min(5, len(order_idx))]
            ]
            dominant_features.append(top_items)

    return {
        'eigvals': eigvals_sorted,
        'eigvecs': eigvecs_sorted,
        'W_r': W_r,
        'z_eig_current': np.real_if_close(z_eig_current),
        'coarse_grain_matrix': np.real_if_close(coarse_grain_matrix),
        'dominant_features': dominant_features,
    }


def safe_logdet_psd(matrix, eps=1e-10):
    matrix = np.asarray(matrix, dtype=float)
    matrix = 0.5 * (matrix + matrix.T)
    evals = np.linalg.eigvalsh(matrix)
    evals = np.clip(evals, eps, None)
    return float(np.sum(np.log(evals)))


def gaussian_mutual_information(A, B, eps=1e-10):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    if A.ndim == 1:
        A = A[:, None]
    if B.ndim == 1:
        B = B[:, None]
    AB = np.concatenate([A, B], axis=1)
    cov_A = np.cov(A, rowvar=False)
    cov_B = np.cov(B, rowvar=False)
    cov_AB = np.cov(AB, rowvar=False)
    if np.ndim(cov_A) == 0:
        cov_A = np.array([[float(cov_A)]])
    if np.ndim(cov_B) == 0:
        cov_B = np.array([[float(cov_B)]])
    if np.ndim(cov_AB) == 0:
        cov_AB = np.array([[float(cov_AB)]])
    mi = 0.5 * (safe_logdet_psd(cov_A, eps=eps) + safe_logdet_psd(cov_B, eps=eps) - safe_logdet_psd(cov_AB, eps=eps))
    return float(mi)


## Block 16. 团序参量与宏观变量的频谱分析

这一块比较微观团序参量与宏观变量的频谱结构。

对于 Kuramoto 系统，微观层面最自然的集体变量不是原始的 `cos/sin` 嵌入分量，而是每个团的序参量 $r_c(t)$。因此，这里先从 `theta_hist` 中提取每个团的序参量时间序列，再与当前选取的 $r$ 个宏观变量放在同一张频谱图中比较。

图中的约定如下：

- 微观团序参量频谱使用虚线；
- 宏观变量频谱使用实线；
- 对宏观变量先做标准化，再计算 FFT，目的是把不同量纲的宏观分量放到可比尺度上；
- 每条曲线额外标注主峰频率，用于直接观察主导节律是否被粗粒化变量保留下来。

In [ ]:
theta_fit = theta_hist[burn_in_steps::sample_stride].copy()
n_clusters = config['kuramoto']['n_clusters']
N_micro = config['kuramoto']['N']
cluster_size = N_micro // n_clusters
representative_indices = []
micro_signals = []
micro_labels = []
for c in range(n_clusters):
    start = c * cluster_size
    representative_idx = start
    representative_indices.append(representative_idx)
    micro_signals.append(x_data_fit[:-config['lag_steps'], representative_idx])
    micro_labels.append(f'Cluster {c + 1} representative oscillator')

eig_macro = build_macro_from_kbar_eig(
    K_bar=K_bar,
    C00_inv_sqrt=C00_inv_sqrt,
    X=X_pairs,
    r=selected_r,
    feature_names=raw_names,
    center=False,
)
z_eig_current = eig_macro['z_eig_current']
eig_macro_names = [f'eig_z_{i + 1}' for i in range(selected_r)]

macro_spectrum_data = macro['z_current']
macro_spectrum_names = [f'z_{i + 1}' for i in range(selected_r)]

#micro_colors = ['#9ED8DB', '#B8E3C6', '#C7E9E8', '#D6F0E0']
macro_colors = ['#1f77b4', '#5CC8BE']

spectrum_summary = {'micro': [], 'macro': []}
micro_spectra = []
macro_spectra = []

for idx, signal in enumerate(micro_signals):
    freqs, amps = compute_fft_spectrum(signal, dt=dt_fit, remove_dc=True, standardize=False)
    color = macro_colors[idx % len(macro_colors)]
    micro_spectra.append({'freqs': freqs, 'amps': amps, 'color': color, 'label': micro_labels[idx]})
    peak_info = None
    spectrum_summary['micro'].append({'label': micro_labels[idx],'representative_index': representative_indices[idx],'peak_frequency': None if peak_info is None else peak_info['peak_frequency'],
        'peak_amplitude': None if peak_info is None else peak_info['peak_amplitude'],})

for idx in range(selected_r):
    standardized_sig = standardize_for_plot(macro_spectrum_data[:, idx])
    freqs, amps = compute_fft_spectrum(standardized_sig, dt=dt_fit, remove_dc=True, standardize=False)
    color = macro_colors[idx % len(macro_colors)]
    macro_spectra.append({'freqs': freqs, 'amps': amps, 'color': color, 'label': macro_spectrum_names[idx]})
    peak_info = None
    spectrum_summary['macro'].append({'label': macro_spectrum_names[idx],'peak_frequency': None if peak_info is None else peak_info['peak_frequency'],
        'peak_amplitude': None if peak_info is None else peak_info['peak_amplitude'],})

paired_count = min(2, len(micro_spectra), len(macro_spectra))
if paired_count > 0:
    fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
    axes = np.asarray(axes)
    for col in range(2):
        if col < paired_count:
            micro_item = micro_spectra[col]
            macro_item = macro_spectra[col]
            axes[0, col].plot(micro_item['freqs'], micro_item['amps'], linestyle='--', linewidth=1.5, color=micro_item['color'])
            micro_peak = mark_peak(axes[0, col], micro_item['freqs'], micro_item['amps'], micro_item['color'], text_offset=0.05)
            axes[0, col].set_title(f'Cluster {col + 1} / Macro {col + 1}')
            axes[0, col].set_ylabel('Micro amplitude')
            axes[0, col].set_xlim(0, 1.0)
            axes[0, col].set_ylim(0, 1.5)
            axes[0, col].grid(True, linestyle='--', alpha=0.4)
            axes[0, col].legend([f"micro: {micro_item['label']}"])
            spectrum_summary['micro'][col]['peak_frequency'] = None if micro_peak is None else micro_peak['peak_frequency']
            spectrum_summary['micro'][col]['peak_amplitude'] = None if micro_peak is None else micro_peak['peak_amplitude']

            axes[1, col].plot(macro_item['freqs'], macro_item['amps'], linestyle='-', linewidth=1.2, alpha=0.5, color=macro_item['color'])
            macro_peak = mark_peak(axes[1, col], macro_item['freqs'], macro_item['amps'], macro_item['color'], text_offset=0.08)
            axes[1, col].set_ylabel('Macro amplitude')
            axes[1, col].set_xlabel('Frequency (Hz)')
            axes[1, col].set_xlim(0, 1.0)
            axes[1, col].set_ylim(0, 1.5)
            axes[1, col].grid(True, linestyle='--', alpha=0.4)
            axes[1, col].legend([f"macro: {macro_item['label']}"])
            spectrum_summary['macro'][col]['peak_frequency'] = None if macro_peak is None else macro_peak['peak_frequency']
            spectrum_summary['macro'][col]['peak_amplitude'] = None if macro_peak is None else macro_peak['peak_amplitude']
        else:
            axes[0, col].axis('off')
            axes[1, col].axis('off')
    plt.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(9, 4.8))
for idx, item in enumerate(micro_spectra):
    ax.plot(item['freqs'], item['amps'], linestyle='--', linewidth=1.5, color=item['color'], label=f"micro: {item['label']}")
    peak_info = mark_peak(ax, item['freqs'], item['amps'], item['color'], text_offset=0.05)
    spectrum_summary['micro'][idx]['peak_frequency'] = None if peak_info is None else peak_info['peak_frequency']
    spectrum_summary['micro'][idx]['peak_amplitude'] = None if peak_info is None else peak_info['peak_amplitude']

for idx, item in enumerate(macro_spectra):
    ax.plot(item['freqs'], item['amps'], linestyle='-', linewidth=1.2, alpha=0.5, color=item['color'], label=f"macro: {item['label']}")
    peak_info = mark_peak(ax, item['freqs'], item['amps'], item['color'], text_offset=0.08)
    spectrum_summary['macro'][idx]['peak_frequency'] = None if peak_info is None else peak_info['peak_frequency']
    spectrum_summary['macro'][idx]['peak_amplitude'] = None if peak_info is None else peak_info['peak_amplitude']

ax.set_title('Spectrum comparison: representative oscillators vs macro variables')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Amplitude')
ax.set_xlim(0, 1.0)
ax.set_ylim(0, 1.5)
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

artifacts['macro']['eig_macro'] = eig_macro
artifacts['spectral']['cluster_spectrum_summary'] = spectrum_summary
artifacts['spectral']['micro_representative_indices'] = representative_indices

print('Micro representative oscillator spectrum peaks:')
display(pd.DataFrame(spectrum_summary['micro']))
print('Macro spectrum peaks:')
display(pd.DataFrame(spectrum_summary['macro']))


## Block 17. 高斯多元互信息比较

这一块用高斯多元互信息来比较不同层次变量对未来微观状态的信息保留能力。

设两个连续多维随机变量分别为 $A$ 和 $B$，如果把它们近似看作联合高斯变量，则互信息可以写成协方差矩阵的对数行列式形式：

$$
I(A;B) = \frac12 \log \frac{\det \Sigma_A \, \det \Sigma_B}{\det \Sigma_{AB}}.
$$

这里选择这种定义有两个原因：

- 当前 notebook 的主链本身就是围绕协方差、白化和谱分解展开的，因此高斯多元互信息与前面的统计结构保持一致；
- 我们关心的是不同宏观表示对未来微观状态 $X_{t+1}$ 的信息保留量，高斯互信息能够直接处理多维连续变量，不需要额外做离散分箱。

本块比较三种量：

1. 微观基准：$I(X_t; X_{t+1})$；
2. SVD 宏观：$I(Y_t; X_{t+1})$，其中 $Y_t$ 取当前主方法构造的宏观变量；
3. Eig 宏观：$I(Z_t; X_{t+1})$，其中 $Z_t$ 来自对 $\bar K$ 直接做特征值分解的对照宏观变量。

其中，特征值分解宏观变量的构造规则为：

- 对 $\bar K$ 直接做特征值分解；
- 按特征值模长从大到小手动排序；
- 对特征向量逐列取模长，使其变成实值非负权重；
- 再把白化后的当前微观变量投影到这些方向上，得到 $Z_t$。

最后用柱状图展示三种互信息，并标出相对于微观基准的信息保留比例。

In [ ]:
mi_x_to_xnext = gaussian_mutual_information(X_pairs, Y_pairs, eps=config['eps'])
mi_z_to_xnext = gaussian_mutual_information(macro['z_current'], Y_pairs, eps=config['eps'])
mi_z_eig_to_xnext = gaussian_mutual_information(z_eig_current, Y_pairs, eps=config['eps'])

mi_ratio_z = mi_z_to_xnext / mi_x_to_xnext if mi_x_to_xnext > 0 else np.nan
mi_ratio_z_eig = mi_z_eig_to_xnext / mi_x_to_xnext if mi_x_to_xnext > 0 else np.nan

mi_summary = pd.DataFrame([
    {
        'name': 'SVD macro to future micro',
        'symbol': r'I(Z_t; X_{t+1})',
        'value': mi_z_to_xnext,
        'ratio_to_micro': mi_ratio_z,
    },
    {
        'name': 'Eig macro to future micro',
        'symbol': r'I(Z_eig_t; X_{t+1})',
        'value': mi_z_eig_to_xnext,
        'ratio_to_micro': mi_ratio_z_eig,
    },
    {
        'name': 'Micro to future micro',
        'symbol': r'I(X_t; X_{t+1})',
        'value': mi_x_to_xnext,
        'ratio_to_micro': 1.0,
    },
])

display(mi_summary)

fig, ax = plt.subplots(figsize=(3, 4))
color_total = '#E5E8E8'
color_sub1 = '#004488'
color_sub2 = '#4682B4'

total_value = mi_x_to_xnext
ratio_values = [mi_ratio_z, mi_ratio_z_eig]
raw_values = [mi_z_to_xnext, mi_z_eig_to_xnext]
sub_labels = [r'SVD', r'EVD']

center_x = 0.0
offset = 0.18
sub_x = np.array([center_x - offset, center_x + offset])

total_bar = ax.bar([center_x],[1.0],width=0.62,color=color_total,edgecolor='none',zorder=1,)
sub_bars = ax.bar(sub_x,ratio_values,width=0.24,color=[color_sub1, color_sub2],edgecolor='none',zorder=3,)

total_height = total_bar[0].get_height()
ax.text(center_x,total_height,f'{total_value:.4f}\n(100.0%)',ha='center',va='bottom',fontsize=10,fontweight='bold',color=color_sub1,)

for bar, raw_value, ratio in zip(sub_bars, raw_values, ratio_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2.0,height,f'{raw_value:.4f}\n({ratio * 100:.1f}%)',ha='center',va='bottom',fontsize=10,fontweight='bold',color=color_sub1,)

ax.set_xticks(sub_x)
ax.set_xticklabels(sub_labels, fontsize=10)

ax.set_title('Gaussian multivariate mutual information comparison')
ax.set_ylabel('Ratio relative to micro mutual information')
ax.set_ylim(0, max(1.12, np.nanmax([1.0] + ratio_values) + 0.12))
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

artifacts['metrics']['mi_summary'] = mi_summary
artifacts['metrics']['mi_x_to_xnext'] = float(mi_x_to_xnext)
artifacts['metrics']['mi_z_to_xnext'] = float(mi_z_to_xnext)
artifacts['metrics']['mi_z_eig_to_xnext'] = float(mi_z_eig_to_xnext)
artifacts['metrics']['mi_ratio_z'] = float(mi_ratio_z)
artifacts['metrics']['mi_ratio_z_eig'] = float(mi_ratio_z_eig)

print('Top eigenvalue magnitudes of K_bar:', np.abs(eig_macro['eigvals'][: min(10, len(eig_macro['eigvals']))]))
print('Mutual information summary:')
print(mi_summary.to_string(index=False))
